In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE

from ugdatalab.models.galaxy_zoo.constants import (
    LABEL_COLUMNS,
    LABEL_DESCRIPTIVE,
)
from ugdatalab.models.galaxy_zoo.images import _load_image

import plotters

# Galaxy Image Classification — Diagnostic Plots

This notebook produces six diagnostic plots that build on the label distributions, prototype images, and correlation matrix from NB 01. The goals are to:

1. Embed the 37-dimensional label space into 2D via t-SNE
2. Show label-conditional image montages for strongly correlated label pairs
3. Plot pairwise label scatter with density coloring
4. Quantify effective sample size per label
5. Compute the PCA eigenspectrum of the label correlation matrix
6. Compare pixel intensity statistics between smooth and disk galaxies

In [ ]:
data = np.load("artifacts/galaxy_zoo_labels.npz")
labels = data["labels"]
galaxy_ids = data["galaxy_ids"]
corr_matrix = data["corr_matrix"]

label_desc_list = [LABEL_DESCRIPTIVE[col] for col in LABEL_COLUMNS]

print(f"Labels: {labels.shape}")
print(f"Correlation matrix: {corr_matrix.shape}")

## Diagnostic 1 — t-SNE of the 37-Dimensional Label Space

We embed each galaxy's 37-dimensional label vector into 2D using t-SNE (perplexity 30, 1000 iterations). Points are colored by the Smooth vote fraction (Class1.1): red = smooth, blue = features/disk. This reveals whether the label space has natural clusters, continuous gradients, or isolated outlier populations.

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42)
embedding = tsne.fit_transform(labels)
print(f"t-SNE embedding: {embedding.shape}")

In [ ]:
smooth_fraction = labels[:, LABEL_COLUMNS.index("Class1.1")]
ax = plotters.plot_label_tsne(embedding, smooth_fraction, "Smooth fraction")
plt.show()

## Diagnostic 2 — Label-Conditional Image Montages

For two strongly correlated label pairs — (Spiral: Yes, Medium winding) and (Features/Disk, Edge-on: No) — we display $4 \times 3$ image grids showing the four quadrants: high-high, high-low, low-high, and low-low. This visually confirms whether the correlations correspond to morphologically distinct populations or are artifacts of the hierarchical gating.

In [ ]:
IMAGE_DIR = Path("data/training_images")
img_data = np.load("artifacts/galaxy_zoo_images.npz")
images = img_data["images"]
print(f"Images: {images.shape}")

In [ ]:
spiral_yes = labels[:, LABEL_COLUMNS.index("Class4.1")]
medium_wind = labels[:, LABEL_COLUMNS.index("Class10.2")]

axes = plotters.plot_label_conditional_montage(
    images, spiral_yes, medium_wind,
    "Spiral: Yes", "Medium winding",
    0.5, 42,
)
plt.show()

In [ ]:
features_disk = labels[:, LABEL_COLUMNS.index("Class1.2")]
edge_on_no = labels[:, LABEL_COLUMNS.index("Class2.2")]

axes = plotters.plot_label_conditional_montage(
    images, features_disk, edge_on_no,
    "Features/Disk", "Edge-on: No",
    0.5, 42,
)
plt.show()

## Diagnostic 3 — Pairwise Scatter Plots of Correlated Labels

We plot the 8 most informative label pairs (4 strongest positive, 4 strongest negative correlations, excluding trivially constrained siblings). Points are colored by local density to reveal the shape of the bivariate distribution — linear, bimodal, or gated.

In [ ]:
pairs = [
    # Strong positive
    (LABEL_COLUMNS.index("Class2.1"), LABEL_COLUMNS.index("Class9.1")),   # Edge-on → Rounded bulge
    (LABEL_COLUMNS.index("Class4.1"), LABEL_COLUMNS.index("Class10.2")),  # Spiral → Medium winding
    (LABEL_COLUMNS.index("Class2.2"), LABEL_COLUMNS.index("Class4.1")),   # Not edge-on → Spiral
    (LABEL_COLUMNS.index("Class4.1"), LABEL_COLUMNS.index("Class11.2")),  # Spiral → 2 arms
    # Strong negative / cross-branch
    (LABEL_COLUMNS.index("Class1.1"), LABEL_COLUMNS.index("Class1.2")),   # Smooth vs Features
    (LABEL_COLUMNS.index("Class1.1"), LABEL_COLUMNS.index("Class4.1")),   # Smooth vs Spiral
    (LABEL_COLUMNS.index("Class1.2"), LABEL_COLUMNS.index("Class7.1")),   # Features vs Round
    (LABEL_COLUMNS.index("Class7.3"), LABEL_COLUMNS.index("Class2.1")),   # Cigar vs Edge-on
]

axes = plotters.plot_pairwise_label_scatter(labels, pairs, label_desc_list)
plt.show()

## Diagnostic 4 — Effective Sample Size per Label

Since deeper labels are gated by parent answers, the number of galaxies with meaningful (nonzero) values varies dramatically. We define $N_{\text{eff}}$ as the count of galaxies with label value $> 0.1$ and plot it as a horizontal bar chart. Labels with small $N_{\text{eff}}$ will have the poorest training signal and highest per-label RMSE.

Colors: blue = $N_{\text{eff}} > 1000$ (well-sampled), orange = $200$–$1000$ (marginal), red = $< 200$ (severely undersampled).

In [ ]:
ax = plotters.plot_effective_sample_size(labels, label_desc_list, 0.1)
plt.show()

## Diagnostic 5 — PCA Eigenspectrum of the Label Correlation Matrix

We compute the eigenvalues of the $37 \times 37$ label correlation matrix. The number of eigenvalues above 1 (Kaiser criterion) gives the effective dimensionality of the label space. The cumulative variance panel shows how many components are needed to capture 90\% of the total variance — likely far fewer than 37 given the hierarchical constraints.

In [ ]:
eigenvalues = np.linalg.eigvalsh(corr_matrix)[::-1]  # descending
n_kaiser = np.sum(eigenvalues > 1)
cumvar = np.cumsum(eigenvalues) / np.sum(eigenvalues)
n_90 = np.searchsorted(cumvar, 0.9) + 1

print(f"Eigenvalues above Kaiser criterion: {n_kaiser}")
print(f"Components for 90% variance: {n_90}")
print(f"Top 5 eigenvalues: {eigenvalues[:5]}")

axes = plotters.plot_pca_eigenspectrum(eigenvalues)
plt.show()

## Diagnostic 6 — Pixel Intensity Statistics by Morphology

We compute per-image statistics — mean brightness, standard deviation (contrast), and red-to-blue color ratio — and compare smooth vs. disk galaxies. Smooth ellipticals tend to be redder and more centrally concentrated than blue spirals. If this separation is visible in raw pixel statistics, the CNN may exploit a simple color/brightness shortcut for the top-level classification.

In [ ]:
smooth_mask = labels[:, LABEL_COLUMNS.index("Class1.1")] > 0.5
disk_mask = labels[:, LABEL_COLUMNS.index("Class1.2")] > 0.5

print(f"Smooth galaxies (Class1.1 > 0.5): {smooth_mask.sum()}")
print(f"Disk galaxies (Class1.2 > 0.5): {disk_mask.sum()}")

# Compute per-image statistics
images_float = images.astype(np.float32) / 255.0 if images.dtype == np.uint8 else images.astype(np.float32)

mean_brightness = np.mean(images_float, axis=(1, 2, 3))
std_brightness = np.std(images_float, axis=(1, 2, 3))
# Red (i-band) to blue (g-band) ratio: channel 0 is R, channel 2 is B
mean_r = np.mean(images_float[:, :, :, 0], axis=(1, 2))
mean_b = np.mean(images_float[:, :, :, 2], axis=(1, 2))
color_ratio = np.where(mean_b > 0, mean_r / mean_b, 1.0)

stats_smooth = {
    "Mean brightness": mean_brightness[smooth_mask],
    "Contrast (std)": std_brightness[smooth_mask],
    "Red/Blue ratio": color_ratio[smooth_mask],
}
stats_disk = {
    "Mean brightness": mean_brightness[disk_mask],
    "Contrast (std)": std_brightness[disk_mask],
    "Red/Blue ratio": color_ratio[disk_mask],
}

axes = plotters.plot_pixel_statistics(
    stats_smooth, stats_disk,
    ["Mean brightness", "Contrast (std)", "Red/Blue ratio"],
)
plt.show()